# MultiMNIST — validated configuration selection and three-seed training

Choose **Runtime → Change runtime type → A100 GPU**, then **Run all**.

This version selects the learning rate and momentum injection for **Entropic LMO-MGDA** using a fixed validation subset of training data. It then trains the selected configuration from scratch on the entire training set with seeds 42, 43 and 44. The method's oracle and update equations are unchanged. MOON keeps its released settings.

The model learning rate (`--lr`) is distinct from the entropic weight-update step (`--eta`). The previous notebook reused the common MOON learning rate for the differently scaled LMO direction; this version calibrates the model step and momentum injection on validation.

All subprocess logs appear live, with **one metric log at the end of each epoch and no batch progress bars**. Checkpoints and console logs are saved to Drive. Rerun this notebook after a disconnection to resume.

**Compute budget:** default tuning screens 8 configurations for 40 epochs and continues the top 2 to 100 epochs (440 epoch-equivalents in total). Final training adds 3 × 100 epochs. Completed work is reused. Optional baselines and ablations are disabled.

The previous experiment remains under `multimnist_a100_v1`; these runs use a new `multimnist_a100_v2` directory. The existing dataset archive is reused so the images remain the same. Exact reproduction of the paper's reported accuracies is not guaranteed because its generated dataset was not released.


In [3]:
#@title 1. Experiment settings
REPO_URL = "https://github.com/alirezamirrokni/LMO-MOO.git"
REPO_COMMIT = "f6e16f360b792db58763f994013428e6a1a4b5c2"
EXPERIMENT_NAME = "multimnist_a100_v2" #@param {type:"string"}
RUN_TUNING = True #@param {type:"boolean"}
RUN_EXTRA_BASELINES = False #@param {type:"boolean"}
RUN_ABLATIONS = False #@param {type:"boolean"}
EXTRA_METHODS = ["famo", "moon"]
SEEDS = [42, 43, 44]
EPOCHS = 100
BATCH_SIZE = 256
# Keep the existing v1 dataset; these are input files, not old model checkpoints.
DATA_CACHE_EXPERIMENT = "multimnist_a100_v1"


In [4]:
#@title 2. Check A100, mount Drive and enable live console logs
import os, sys, json, subprocess, shutil, time, hashlib, zipfile, csv
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > A100 GPU."
GPU_NAME = torch.cuda.get_device_name(0)
assert "A100" in GPU_NAME, f"Current GPU: {GPU_NAME}. Select A100 and reconnect."
print("GPU:", GPU_NAME, "| PyTorch:", torch.__version__)
drive.mount("/content/drive")
assert EXPERIMENT_NAME and Path(EXPERIMENT_NAME).name == EXPERIMENT_NAME and EXPERIMENT_NAME not in {".", ".."}
DRIVE_ROOT = Path("/content/drive/MyDrive/LMO-MOO")
RUN_ROOT = DRIVE_ROOT / EXPERIMENT_NAME
OUTPUT_ROOT = RUN_ROOT / "results" / "multimnist"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO = Path("/content/LMO-MOO-v2")
DATA = Path("/content/LMO-MOO-data/multimnist")
print("Persistent outputs:", OUTPUT_ROOT)


import codecs, signal, shlex, uuid
LOG_DIR = RUN_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)

def run_live(cmd, *, cwd=None, label="process"):
    """Forward stdout/stderr chunks immediately, preserving carriage returns.

    Raw console logs are also saved to Drive. An interrupted cell stops the
    entire subprocess group, including children launched by the suite.
    """
    cmd = list(map(str, cmd))
    print("$ " + shlex.join(cmd), flush=True)
    safe_label = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in label)
    log_path = LOG_DIR / (time.strftime("%Y%m%d_%H%M%S") + "_" + safe_label + "_" + uuid.uuid4().hex[:6] + ".log")
    env = os.environ.copy()
    env.update(PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
    print("Console log:", log_path, flush=True)
    with log_path.open("wb", buffering=0) as log:
        process = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, bufsize=0,
                                   start_new_session=True)
        decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
        try:
            while True:
                chunk = os.read(process.stdout.fileno(), 8192)
                if not chunk:
                    break
                # Display first so a slow Drive write cannot hide current output.
                sys.stdout.write(decoder.decode(chunk))
                sys.stdout.flush()
                log.write(chunk)
            sys.stdout.write(decoder.decode(b"", final=True))
            sys.stdout.flush()
            returncode = process.wait()
        except BaseException:
            if process.poll() is None:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
            raise
        finally:
            process.stdout.close()
    print(f"\n[{label}] Exit code: {returncode}", flush=True)
    if returncode:
        raise subprocess.CalledProcessError(returncode, cmd)
    return log_path


GPU: NVIDIA A100-SXM4-80GB | PyTorch: 2.11.0+cu128
Mounted at /content/drive
Persistent outputs: /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/results/multimnist


In [5]:
#@title 3. Clone the corrected code and install dependencies
if not REPO.exists():
    run_live(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "--sparse", "--progress", REPO_URL, REPO], label="clone")
else:
    assert (REPO / ".git").exists(), f"{REPO} is not a Git checkout."
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    assert origin == REPO_URL
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True)
    assert not dirty, "Save local tracked-file edits before rerunning setup."
if subprocess.run(["git", "-C", str(REPO), "cat-file", "-e", REPO_COMMIT + "^{commit}"], capture_output=True).returncode:
    run_live(["git", "-C", REPO, "fetch", "origin", REPO_COMMIT], label="fetch")
run_live(["git", "-C", REPO, "sparse-checkout", "set", "experiments", "methods", "scripts"], label="checkout-files")
run_live(["git", "-C", REPO, "checkout", "--detach", REPO_COMMIT], label="checkout-revision")
# Preserve Colab's installed CUDA-compatible torch and torchvision pair.
run_live([sys.executable, "-m", "pip", "install", "-r", REPO / "requirements-modern.txt", "pandas"], label="dependencies")
run_live([sys.executable, "-u", "-c", "import torch, torchvision; from experiments.multimnist.trainer import parser; print('Imports OK:', torch.__version__, torchvision.__version__)"], cwd=REPO, label="import-check")
os.chdir(REPO)
def run_script(filename, *args):
    return run_live([sys.executable, "-u", REPO / filename, *args], cwd=REPO, label=Path(filename).stem)

def atomic_json(path, value):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, indent=2) + "\n")
    tmp.replace(path)

identity = {"repo": REPO_URL, "commit": REPO_COMMIT, "epochs": EPOCHS,
            "batch_size": BATCH_SIZE, "seeds": SEEDS, "dataset_seed": 2026,
            "train_samples": 10000, "test_samples": 1000}
manifest = RUN_ROOT / "experiment.json"
if manifest.exists():
    assert json.loads(manifest.read_text()) == identity, "Experiment settings changed. Use a new EXPERIMENT_NAME."
else:
    atomic_json(manifest, identity)
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_ROOT / ("environment_" + time.strftime("%Y%m%d_%H%M%S") + ".txt")).write_text(versions)
print("Pinned code revision:", REPO_COMMIT)


$ git clone --depth 1 --filter=blob:none --no-checkout --sparse --progress https://github.com/alirezamirrokni/LMO-MOO.git /content/LMO-MOO-v2
Console log: /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/logs/20260924_183811_clone_f18588.log
Cloning into '/content/LMO-MOO-v2'...
remote: Enumerating objects: 21, done.        
remote: Counting objects: 100% (21/21), done.        
remote: Compressing objects: 100% (20/20), done.        
remote: Total 21 (delta 0), reused 16 (delta 0), pack-reused 0 (from 0)        
Receiving objects: 100% (21/21), 6.08 KiB | 6.08 MiB/s, done.

[clone] Exit code: 0
$ git -C /content/LMO-MOO-v2 sparse-checkout set experiments methods scripts
Console log: /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/logs/20260924_183812_checkout-files_60389f.log

[checkout-files] Exit code: 0
$ git -C /content/LMO-MOO-v2 checkout --detach f6e16f360b792db58763f994013428e6a1a4b5c2
Console log: /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/logs/20260924_183812_checkou

In [6]:
#@title 4. Prepare and cache the fixed dataset
CACHE = DRIVE_ROOT / DATA_CACHE_EXPERIMENT / "data"
CACHE.mkdir(parents=True, exist_ok=True)
ARCHIVE = CACHE / "multimnist_seed2026.zip"
LOCAL_ARCHIVE = Path("/content/multimnist_seed2026.zip")
if not ARCHIVE.exists():
    # Remove only a partial generated dataset from an interrupted preparation.
    if DATA.exists():
        shutil.rmtree(DATA)
    run_script("prepare_multimnist.py", "--download", "--output", DATA,
               "--mnist-root", "/content/LMO-MOO-data/mnist",
               "--train-samples", 10000, "--test-samples", 1000, "--seed", 2026)
    with zipfile.ZipFile(LOCAL_ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(DATA.rglob("*")):
            if p.is_file():
                z.write(p, p.relative_to(DATA))
    print("Saving the dataset archive to Drive...", flush=True)
    partial = ARCHIVE.with_suffix(".zip.partial")
    shutil.copy2(LOCAL_ARCHIVE, partial)
    partial.replace(ARCHIVE)
else:
    print("Restoring the dataset archive from Drive...", flush=True)
    shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)

digest = hashlib.sha256(LOCAL_ARCHIVE.read_bytes()).hexdigest()
hash_file = CACHE / "dataset_archive.sha256"
if hash_file.exists():
    assert hash_file.read_text().strip() == digest, "Dataset archive checksum mismatch."
else:
    hash_file.write_text(digest + "\n")
# Reuse local data within a session; restore it after reconnecting.
marker = DATA.parent / "archive.sha256"
if not (DATA.exists() and marker.exists() and marker.read_text().strip() == digest):
    if DATA.exists():
        shutil.rmtree(DATA)
    DATA.mkdir(parents=True)
    with zipfile.ZipFile(LOCAL_ARCHIVE) as z:
        assert z.testzip() is None, "Damaged dataset archive."
        for info in z.infolist():
            target = (DATA / info.filename).resolve()
            assert target.is_relative_to(DATA.resolve()), "Unsafe archive entry."
        for info in z.infolist():
            z.extract(info, DATA)
    marker.write_text(digest + "\n")
for split, expected in [("train", 10000), ("test", 1000)]:
    with (DATA / split / "labels.csv").open() as f:
        rows = list(csv.reader(f))
    assert len(rows) == expected
    assert all((DATA / split / "2" / row[0]).is_file() for row in rows)
    print(split, len(rows), "images")
print("Dataset ready on local disk:", DATA)


Restoring the dataset archive from Drive...
train 10000 images
test 1000 images
Dataset ready on local disk: /content/LMO-MOO-data/multimnist


In [ ]:
#@title 5. Select the configuration on validation only (resume-safe)
TUNING_ROOT = RUN_ROOT / "tuning"
SELECTION_FILE = TUNING_ROOT / "selection.json"
COMMON = ["--data-path", DATA, "--device", "cuda:0", "--epochs", EPOCHS,
          "--batch-size", BATCH_SIZE, "--workers", 0, "--threads", 4, "--cache-data"]
if RUN_TUNING:
    run_script("run_multimnist_tune.py", "--output-root", TUNING_ROOT, *COMMON)
assert SELECTION_FILE.exists(), "No selected configuration yet. Enable RUN_TUNING and run this cell."
selection = json.loads(SELECTION_FILE.read_text())
assert selection["test_used"] is False
assert selection["settings"]["selection"] == "validation"
SELECTED = selection["selected"]
assert SELECTED["epochs"] == EPOCHS and SELECTED["batch_size"] == BATCH_SIZE
# Verify that selection belongs to the currently restored training images.
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from experiments.multimnist.trainer import data_fingerprint
assert selection["settings"]["train_sha256"] == data_fingerprint(DATA, splits=("train",))
print("Selected configuration:", json.dumps(SELECTED, indent=2))
print("Final validation metrics (not test results):", selection["validation"])
# Prevent accidental mixing of existing final runs with a new selection.
final_manifest = RUN_ROOT / "final_configuration.json"
if final_manifest.exists():
    assert json.loads(final_manifest.read_text()) == SELECTED, "Selection changed. Use a new experiment directory."
else:
    atomic_json(final_manifest, SELECTED)
OURS_FLAGS = ["--lr", SELECTED["lr"], "--eta", SELECTED["eta"], "--alpha", SELECTED["alpha"],
              "--oracle", SELECTED["oracle"], "--weights", SELECTED["weights"], "--momentum", SELECTED["momentum"]]


$ /usr/bin/python3 -u /content/LMO-MOO-v2/run_multimnist_tune.py --output-root /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/tuning --data-path /content/LMO-MOO-data/multimnist --device cuda:0 --epochs 100 --batch-size 256 --workers 0 --threads 4 --cache-data
Console log: /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/logs/20260924_183832_run_multimnist_tune_2cf2d7.log

Screen 1/8: lr=0.001, alpha=0.1

/usr/bin/python3 -u /content/LMO-MOO-v2/run_multimnist.py --method ours --seed 42 --lr 0.001 --eta 0.0001 --alpha 0.1 --selection validation --metric sample --split-seed 2026 --val-fraction 0.1 --epochs 100 --stop-after-epoch 40 --batch-size 256 --data-path /content/LMO-MOO-data/multimnist --output-root /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/tuning --tag lr-0.001-alpha-0.1 --resume --device cuda:0 --workers 0 --threads 4 --cache-data
Run: method=ours, seed=42, selection=validation, lr=0.001, eta=0.0001, alpha=0.1
[MultiMNISTDataset] split=train, num_samples=10000, img_d

The following cell starts fresh final models on **all 10,000 training images**, using the selected hyperparameters. It never loads a tuning checkpoint. Subsequent reruns resume only these final runs. At the end of each epoch, the log reports train/test accuracies, training losses, task weights, inner gap and elapsed seconds.


In [ ]:
#@title 6. Train Entropic LMO-MGDA on all training data with three seeds
started = time.monotonic()
run_script("run_multimnist_suite.py", "--suite", "main", "--methods", "ours",
           "--seeds", *SEEDS, "--output-root", OUTPUT_ROOT, *COMMON, *OURS_FLAGS)
print(f"This cell took {(time.monotonic() - started) / 60:.1f} minutes.")
print("Saved results:", OUTPUT_ROOT)


In [ ]:
#@title 7. Optional baselines and ablations
if RUN_EXTRA_BASELINES:
    # No tuned LMO parameters are passed to baselines. Their released defaults remain intact.
    run_script("run_multimnist_suite.py", "--suite", "main", "--methods", *EXTRA_METHODS,
               "--seeds", *SEEDS, "--output-root", OUTPUT_ROOT, *COMMON)
if RUN_ABLATIONS:
    # All other settings use the same selected base configuration.
    run_script("run_multimnist_suite.py", "--suite", "ablations", "--seeds", *SEEDS,
               "--output-root", OUTPUT_ROOT, *COMMON, *OURS_FLAGS)
if not RUN_EXTRA_BASELINES and not RUN_ABLATIONS:
    print("Optional baselines and ablations are disabled.")


In [ ]:
#@title 8. Generate result tables and LaTeX
from IPython.display import display
import pandas as pd
REPORT = OUTPUT_ROOT / "report"
# This table uses the paper's baseline numbers; these are NOT local reruns.
run_script("report_multimnist.py", "--root", OUTPUT_ROOT, "--seeds", *SEEDS,
           "--baseline-source", "reported", "--out", REPORT)
df = pd.read_csv(REPORT / "results.csv")
display(df[["tag", "method", "n_seeds", "left", "right", "avg", "avg_std", "gap"]])
print("Measured results CSV:", REPORT / "results.csv")
print("LaTeX (published baselines + measured ours):", REPORT / "table.tex")
# A second table contains only locally measured results; missing methods show --.
run_script("report_multimnist.py", "--root", OUTPUT_ROOT, "--seeds", *SEEDS,
           "--baseline-source", "reproduced", "--out", OUTPUT_ROOT / "report_local_only")


**Files in Drive:** `MyDrive/LMO-MOO/multimnist_a100_v2/` (or your chosen experiment name).

- `tuning/search.json` records the entire search protocol; `tuning/selection.json` records all screening/finalist scores and the selected configuration. These are validation results.
- `results/multimnist/main/ours/seed42`, `seed43`, `seed44` contain final training checkpoints and test metrics.
- `results/multimnist/report/results.csv` contains completed local measurements. `report/table.tex` combines measured ours with the paper's reported baselines; `report_local_only/table.tex` contains only local measurements.
- `logs/` contains the live console logs, including errors. No batch progress-bar patch is required.
- To resume after a disconnection, run all cells with unchanged settings. An unfinished epoch may repeat. Stop the old notebook before starting this one.
- Keep optional ablations disabled until the selected main run has finished. To run all baselines, set `EXTRA_METHODS = ["mgda", "mgda_muon", "famo", "famo_muon", "muon_ls", "moon"]` and enable `RUN_EXTRA_BASELINES`.
- Release the GPU when finished using **Runtime → Disconnect and delete runtime**. Drive outputs remain available.
